# 01 · SQL Detective — when does a JOIN multiply records?

**Goal:** Explore data, identify business keys and inspect a query plan.

**Setup:** Python + standard-library SQLite; Matplotlib is optional.

**CS bridge:** Relational algebra, B-trees and query planners.

**Important:** SQLite `EXPLAIN QUERY PLAN` is not PostgreSQL `EXPLAIN ANALYZE`; the PostgreSQL lab tests that database separately.

## Watch and learn — Harvard CS50 SQL

[**Watch the original video on YouTube**](https://www.youtube.com/watch?v=WXk7yDqsKxs). Suggested: a focused 5–15 minute segment.

[![Video thumbnail](https://img.youtube.com/vi/WXk7yDqsKxs/hqdefault.jpg)](https://www.youtube.com/watch?v=WXk7yDqsKxs)

Jupyter/VS Code may embed the video with internet access; GitHub may not. All exercises work without it.

In [1]:
# Optional video player (does not download or store the video)
try:
    from IPython.display import display, YouTubeVideo
    display(YouTubeVideo("WXk7yDqsKxs", width=720, height=405))
except ImportError:
    print("Open the YouTube link in the Markdown above.")

**Additional video — CS50 SQL:** [Open the original video on YouTube](https://www.youtube.com/watch?v=WXk7yDqsKxs). Videos may not load in GitHub's preview; the notebook exercises remain self-contained.

## 1. Two tables, minimal data

Before running the next cell, predict what happens if every event has two tags. Why might `COUNT(*)` after a JOIN be misleading?

In [2]:
import sqlite3
conn=sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE events(event_id TEXT PRIMARY KEY, region TEXT NOT NULL, magnitude REAL NOT NULL);
CREATE TABLE tags(event_id TEXT NOT NULL, label TEXT NOT NULL);
INSERT INTO events VALUES ('e1','North',2.5),('e2','South',4.0),('e3','North',3.1);
INSERT INTO tags VALUES ('e1','reviewed'),('e1','priority'),('e2','priority');
""")
print(conn.execute("SELECT COUNT(*) FROM events").fetchone()[0])

3


## 2. Classic cardinality failure

How many rows will the JOIN produce? How many *unique business events* actually exist?

In [3]:
rows=conn.execute("SELECT e.event_id,t.label FROM events e LEFT JOIN tags t ON t.event_id=e.event_id ORDER BY e.event_id").fetchall()
print(rows)
count_joined = len(rows)
unique_events = conn.execute("SELECT COUNT(DISTINCT event_id) FROM events").fetchone()[0]
print("joined rows:",count_joined, "actual events:",unique_events)
assert count_joined == 4 and unique_events == 3

[('e1', 'priority'), ('e1', 'reviewed'), ('e2', 'priority'), ('e3', None)]
joined rows: 4 actual events: 3


## 3. Query plan before and after an index

Did SQLite's planner use an index? Inspect `EXPLAIN QUERY PLAN`. Do not assume an index always improves performance: scanning a very small table can be cheaper.

In [4]:
query="SELECT * FROM events WHERE region = 'North'"
print("before:",conn.execute("EXPLAIN QUERY PLAN "+query).fetchall())
conn.execute("CREATE INDEX idx_events_region ON events(region)")
print("after:",conn.execute("EXPLAIN QUERY PLAN "+query).fetchall())
assert len(conn.execute(query).fetchall()) == 2

before: [(2, 0, 0, 'SCAN events')]
after: [(3, 0, 0, 'SEARCH events USING INDEX idx_events_region (region=?)')]


## 4. Transfer exercise

Write a `GROUP BY` query that returns the **distinct event count per region**, even when the `tags` table has 0–N tags per event. Explain why `COUNT(DISTINCT e.event_id)` differs from `COUNT(*)`.

**Failure injection:** Add two tags to `e2`, then identify which query results should change and which should remain invariant.

**Next:** [Stage 01](../curriculum/modules/01.md) → [Notebook 02](02_etl_failure_lab.ipynb).